# UCSD Greynet Telescope — pcap → FlowTuple, and a first empirical look

`greynet_data_access.ipynb` found that `telescope-sdscnt-pcap-live` holds **raw packet
captures** (`gzip -> pcap (usec LE)`), not FlowTuple, e.g.

```
datasource=sdsc-nt/year=2025/month=02/day=20/hour=20/sdscnt.1740082500.pcap.gz
```

so the flow-aggregation step lives with us. This notebook does it:

1. lists the pcaps under one S3 prefix (§3),
2. **streams** each one (S3 body → gzip → pcap records; nothing is downloaded to disk or
   held whole in memory) and aggregates packets into **FlowTuple records** (§4–5),
3. writes one Parquet file per pcap (§5), and
4. reads the Parquet back and reports on it: unique source / destination addresses,
   protocol mix, TCP destination and source ports, TCP flags, UDP ports, ICMP types,
   top sources, destination coverage, TTLs and packet sizes (§6–12).

### Which "FlowTuple"

A FlowTuple record (corsaro) is one row per distinct key per time interval, with a packet
count:

| field | meaning |
|---|---|
| `time` | start of the interval (unix seconds; 60 s bins by default, as corsaro uses) |
| `src_ip`, `dst_ip` | IPv4 addresses as unsigned 32-bit integers (as in the Avro FlowTuple) |
| `src_port`, `dst_port` | TCP/UDP ports; for ICMP, **type** and **code** |
| `protocol` | IP protocol number (6 TCP, 17 UDP, 1 ICMP, …) |
| `ttl`, `tcp_flags`, `ip_len` | IP TTL, TCP flag byte, IP total length (wire length, so unaffected by snaplen) |
| `tcp_synlen`, `tcp_synwinlen` | TCP header length and window of the flow's first SYN (0 otherwise) |
| `packet_cnt` | packets that matched this key in this interval |
| `is_masscan` | SYN carries masscan's signature `ip_id == (dst_ip ^ dst_port ^ seq) & 0xffff` |

This is the **per-destination (v3-style) layout**, chosen on purpose: the FlowTuple *v4*
records the darknet module reads have already rolled destinations and source ports up into
`dst_net` + `uniq_*` counters, which would make the destination-address and source-port
questions below unanswerable. v4 is a `groupby` away from this table; the reverse is not
possible.

**Not produced here** (they need external datasets, not the pcap): `prefix2asn`,
`netacq_country` / `maxmind_country`, and `is_spoofed`. The darknet module's
`build_prefix_to_asn` / GeoLite2 enrichment in `nids-telescope-traffic` is the pattern to
add them.

Runs on **Expanse**; read-only against the object store; prints no credential values.

## 0 · Dependencies

Same import-then-`pip --user` pattern as `greynet_data_access.ipynb`. `dpkt` is used only
for its streaming pcap record reader; headers are unpacked by hand (§4).

In [ ]:
import importlib
import site
import subprocess
import sys


def ensure(pkg, module=None):
    "Import `module` (default: pkg), pip-installing --user first if missing."
    module = module or pkg
    try:
        importlib.import_module(module)
        print(f"{pkg:<10} OK")
        return
    except ImportError:
        pass
    print(f"{pkg:<10} missing — installing with pip --user ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--quiet", pkg])
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())
    importlib.invalidate_caches()
    importlib.import_module(module)
    print(f"{pkg:<10} installed")


ensure("boto3")
ensure("dpkt")
ensure("pandas")
ensure("pyarrow")
ensure("matplotlib")

## 1 · Configuration

`PREFIX` selects which pcaps to convert; the bucket is partitioned
`datasource=/year=/month=/day=/hour=/`, one pcap per 5-minute window, so an hour prefix
holds up to 12. `MAX_PCAPS` caps the run — start with 1 to see how long a file takes
(the per-file packets/s is printed), then raise it.

`LOCAL_PCAPS` bypasses S3 entirely: put paths to `.pcap` / `.pcap.gz` files there to run
the same pipeline on local captures.

In [ ]:
ENV_FILE = "/expanse/lustre/projects/sdp178/kmok/.ucsdnts3greynet.env"

BUCKET = "telescope-sdscnt-pcap-live"
PREFIX = "datasource=sdsc-nt/year=2025/month=02/day=20/hour=20/"
MAX_PCAPS = 1            # pcaps to convert, in key (= time) order; None = all under PREFIX

LOCAL_PCAPS = []         # e.g. ["/scratch/.../sdscnt.1740082500.pcap.gz"]; non-empty => S3 is skipped

INTERVAL = 60            # FlowTuple time bin, seconds
OUT_DIR = "flowtuples"   # one <pcap name>.flowtuple.parquet per pcap
OVERWRITE = False        # False: a pcap whose Parquet already exists is not re-parsed

TOP_N = 20               # rows in the "top ..." tables

# Same gateway and override rules as greynet_data_access.ipynb.
DEFAULT_ENDPOINT = "https://hermes.caida.org"
ENDPOINT_OVERRIDE = None

## 2 · Credentials and S3 client

Identical to `greynet_data_access.ipynb` §2–3: parse the shell-style env file,
auto-detect the access / secret / endpoint variables, print only their **names**.
Skipped when `LOCAL_PCAPS` is set.

In [ ]:
import os
import pathlib
import re


def parse_env_file(path):
    "Parse a shell-style env file: optional `export`, KEY=value, quotes, # comments."
    env = {}
    for line in pathlib.Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        line = re.sub(r"^export\s+", "", line)
        m = re.match(r"([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$", line)
        if not m:
            continue
        key, val = m.group(1), m.group(2).strip()
        if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
            val = val[1:-1]
        env[key] = val
    return env


def pick(env, *patterns):
    "First env key matching any regex, tried in order (case-insensitive)."
    for pat in patterns:
        hits = [k for k in env if re.search(pat, k, re.I)]
        if hits:
            return hits[0]
    return None


s3 = None
if LOCAL_PCAPS:
    print("LOCAL_PCAPS set — skipping S3.")
else:
    import boto3
    from botocore.config import Config

    try:
        env = parse_env_file(ENV_FILE)
        print(f"parsed {ENV_FILE}")
    except OSError as exc:
        print(f"could not read {ENV_FILE} ({exc}); falling back to kernel environment")
        env = {k: v for k, v in os.environ.items() if re.search(r"s3|greynet", k, re.I)}

    access_var = pick(env, r"access.?key.?id", r"access.?key", r"access")
    secret_var = pick(env, r"secret.?access.?key", r"secret.?key", r"secret")
    endpoint_var = pick(env, r"endpoint", r"\burl\b", r"host")
    assert access_var and secret_var, (
        f"Could not identify access/secret variables among: {sorted(env)} — "
        "set access_var / secret_var manually in this cell."
    )
    S3_ENDPOINT_URL = (
        ENDPOINT_OVERRIDE or (env.get(endpoint_var) if endpoint_var else None) or DEFAULT_ENDPOINT
    )
    print(f"access key variable : {access_var}")
    print(f"secret key variable : {secret_var}  (value not shown)")
    print(f"endpoint            : {S3_ENDPOINT_URL}")

    s3 = boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT_URL,
        aws_access_key_id=env[access_var],
        aws_secret_access_key=env[secret_var],
        config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
    )

## 3 · Select the pcaps

Each source is `(name, size, opener)`; the opener returns a raw byte stream, so the rest
of the notebook does not care whether a capture is an S3 object or a local file.

In [ ]:
import functools


def _s3_open(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"]


sources = []
if LOCAL_PCAPS:
    for path in LOCAL_PCAPS:
        sources.append((os.path.basename(path), os.path.getsize(path), functools.partial(open, path, "rb")))
else:
    objs = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
        objs.extend(o for o in page.get("Contents", []) if ".pcap" in o["Key"])
    objs.sort(key=lambda o: o["Key"])
    total_gib = sum(o["Size"] for o in objs) / 2**30
    print(f"{len(objs)} pcap objects under s3://{BUCKET}/{PREFIX}  ({total_gib:,.2f} GiB compressed)")
    for o in objs[:MAX_PCAPS]:
        sources.append((o["Key"].rsplit("/", 1)[-1], o["Size"], functools.partial(_s3_open, o["Key"])))

assert sources, "No pcaps selected — check PREFIX / LOCAL_PCAPS."
print(f"\nconverting {len(sources)}:")
for name, size, _ in sources:
    print(f"  {size / 2**20:10.1f} MiB  {name}")

## 4 · Packets → FlowTuple records

**Reading.** The byte stream is sniffed for the gzip magic (never trusting `.gz`),
decompressed incrementally, and handed to `dpkt.pcap.Reader`, which yields one
`(timestamp, frame bytes)` pair at a time.

**Parsing.** Only fixed-offset header fields are needed, so they are unpacked directly
with `struct` instead of building dpkt's per-packet object graph — several times faster,
which matters at tens of millions of packets. The link layer is taken from the pcap
header rather than assumed: Ethernet (with any 802.1Q / QinQ tags skipped), raw IP, and
Linux cooked capture are handled; anything else stops with a clear error.

**What is counted, what is not** — every packet lands in exactly one bucket, printed per
file in §5 so nothing disappears silently:

- `ipv4` — aggregated into FlowTuples.
- `non_ipv4` — ARP, IPv6, etc. (tallied by ethertype). IPv6 is out of scope, as in
  FlowTuple.
- `truncated` — capture cut off before the fields we need (snaplen or a malformed header).

Non-first IP fragments carry no transport header; they are kept with ports and flags 0
and counted under `fragments`.

**Memory.** The key includes the time bin, and each time the capture advances into a new
bin the finished bins are flushed to the Parquet file as a row group. Peak memory is one
interval's worth of distinct flows, independent of how long the capture is.

In [ ]:
import collections
import gzip
import io
import struct
import time as _time

import dpkt
import pyarrow as pa
import pyarrow.parquet as pq

FLOWTUPLE_SCHEMA = pa.schema(
    [
        ("time", pa.int64()),
        ("src_ip", pa.uint32()),
        ("dst_ip", pa.uint32()),
        ("src_port", pa.uint16()),
        ("dst_port", pa.uint16()),
        ("protocol", pa.uint8()),
        ("ttl", pa.uint8()),
        ("tcp_flags", pa.uint8()),
        ("ip_len", pa.uint16()),
        ("tcp_synlen", pa.uint16()),
        ("tcp_synwinlen", pa.uint16()),
        ("packet_cnt", pa.int64()),
        ("is_masscan", pa.bool_()),
    ]
)
_KEY_COLS = FLOWTUPLE_SCHEMA.names[:9]
_VAL_COLS = FLOWTUPLE_SCHEMA.names[9:]

DLT_EN10MB, DLT_RAW, DLT_RAW_ALT, DLT_LINUX_SLL = 1, 12, 101, 113
_VLAN_ETHERTYPES = (0x8100, 0x88A8, 0x9100)
_IP_HDR = struct.Struct("!HHHBBHII")  # total len, id, flags+frag off, ttl, proto, cksum, src, dst
_TCP_HDR = struct.Struct("!HHIIBBH")  # sport, dport, seq, ack, data offset, flags, window
_UDP_PORTS = struct.Struct("!HH")


def open_pcap_stream(raw):
    "Raw byte stream -> buffered, transparently gunzipped stream."
    buffered = io.BufferedReader(raw, 1 << 20) if not hasattr(raw, "peek") else raw
    if buffered.peek(2)[:2] == b"\x1f\x8b":
        return io.BufferedReader(gzip.GzipFile(fileobj=buffered), 1 << 20)
    return buffered


def _flush(writer, flows):
    "Write a {key: [synlen, synwin, count, masscan]} dict as one Parquet row group."
    if not flows:
        return 0
    keys = list(zip(*flows.keys()))
    vals = list(zip(*flows.values()))
    arrays = [pa.array(col, type=FLOWTUPLE_SCHEMA.field(name).type) for name, col in zip(_KEY_COLS, keys)]
    arrays += [pa.array(col, type=FLOWTUPLE_SCHEMA.field(name).type) for name, col in zip(_VAL_COLS, vals)]
    writer.write_table(pa.Table.from_arrays(arrays, schema=FLOWTUPLE_SCHEMA))
    return len(flows)


def pcap_to_flowtuple(raw, out_path, interval=60):
    "Stream one pcap into a FlowTuple Parquet file. Returns a dict of run statistics."
    stream = open_pcap_stream(raw)
    reader = dpkt.pcap.Reader(stream)
    dlt = reader.datalink()
    if dlt not in (DLT_EN10MB, DLT_RAW, DLT_RAW_ALT, DLT_LINUX_SLL):
        raise ValueError(f"unsupported pcap link type {dlt} — add its header length to pcap_to_flowtuple")

    n_packets = n_ipv4 = n_truncated = n_fragments = n_records = wire_bytes = 0
    non_ipv4 = collections.Counter()
    first_ts = last_ts = None
    flows = {}
    cur_bin = None
    unpack_ip, unpack_tcp, unpack_udp = _IP_HDR.unpack_from, _TCP_HDR.unpack_from, _UDP_PORTS.unpack_from

    tmp_path = f"{out_path}.partial"
    t0 = _time.time()
    with pq.ParquetWriter(tmp_path, FLOWTUPLE_SCHEMA, compression="zstd") as writer:
        for ts, buf in reader:
            n_packets += 1
            try:
                # --- link layer -> offset of the IP header ---
                if dlt == DLT_EN10MB:
                    etype = (buf[12] << 8) | buf[13]
                    off = 14
                    while etype in _VLAN_ETHERTYPES:
                        etype = (buf[off + 2] << 8) | buf[off + 3]
                        off += 4
                elif dlt == DLT_LINUX_SLL:
                    etype = (buf[14] << 8) | buf[15]
                    off = 16
                else:
                    etype = 0x0800 if buf[0] >> 4 == 4 else 0x86DD if buf[0] >> 4 == 6 else 0
                    off = 0
                if etype != 0x0800:
                    non_ipv4[etype] += 1
                    continue

                # --- IPv4 ---
                vihl = buf[off]
                if vihl >> 4 != 4:
                    n_truncated += 1
                    continue
                ip_len, ip_id, frag, ttl, proto, _, src, dst = unpack_ip(buf, off + 2)
                l4 = off + (vihl & 0x0F) * 4

                # --- transport ---
                sport = dport = flags = synlen = synwin = 0
                masscan = False
                if frag & 0x1FFF:  # non-first fragment: no transport header
                    n_fragments += 1
                elif proto == 6:
                    sport, dport, seq, _, doff, flags, win = unpack_tcp(buf, l4)
                    if flags & 0x02:  # SYN
                        synlen = (doff >> 4) * 4
                        synwin = win
                        masscan = ip_id == (dst ^ dport ^ seq) & 0xFFFF
                elif proto == 17:
                    sport, dport = unpack_udp(buf, l4)
                elif proto == 1:
                    sport, dport = buf[l4], buf[l4 + 1]  # ICMP type, code
            except (IndexError, struct.error):
                n_truncated += 1
                continue

            n_ipv4 += 1
            wire_bytes += ip_len
            ts = float(ts)
            if first_ts is None:
                first_ts = ts
            last_ts = ts

            tbin = int(ts) // interval * interval
            if cur_bin is None:
                cur_bin = tbin
            elif tbin > cur_bin:  # capture moved into a new interval: flush finished ones
                n_records += _flush(writer, flows)
                flows = {}
                cur_bin = tbin

            key = (tbin, src, dst, sport, dport, proto, ttl, flags, ip_len)
            rec = flows.get(key)
            if rec is None:
                flows[key] = [synlen, synwin, 1, masscan]
            else:
                rec[2] += 1
        n_records += _flush(writer, flows)
    os.replace(tmp_path, out_path)  # a killed run never leaves a half-written file under the final name

    elapsed = _time.time() - t0
    return {
        "datalink": dlt,
        "snaplen": reader.snaplen,
        "packets": n_packets,
        "ipv4": n_ipv4,
        "non_ipv4": sum(non_ipv4.values()),
        "non_ipv4_ethertypes": {f"0x{k:04x}": v for k, v in non_ipv4.most_common(5)},
        "truncated": n_truncated,
        "fragments": n_fragments,
        "flowtuple_records": n_records,
        "ipv4_wire_bytes": wire_bytes,
        "first_ts": first_ts,
        "last_ts": last_ts,
        "parse_seconds": round(elapsed, 1),
        "packets_per_sec": round(n_packets / elapsed) if elapsed else None,
    }

## 5 · Run the conversion

One Parquet file per pcap, named after it. With `OVERWRITE = False` a pcap whose Parquet
already exists is skipped, so re-running the notebook to tweak the analysis below costs
nothing.

In [ ]:
import datetime

import pandas as pd

os.makedirs(OUT_DIR, exist_ok=True)


def out_path_for(name):
    stem = re.sub(r"\.pcap(\.gz)?$", "", name)
    return os.path.join(OUT_DIR, f"{stem}.flowtuple.parquet")


def _utc(ts):
    return datetime.datetime.fromtimestamp(ts, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S") if ts else "-"


flowtuple_files = []
run_stats = []
for name, size, opener in sources:
    out_path = out_path_for(name)
    flowtuple_files.append(out_path)
    if os.path.exists(out_path) and not OVERWRITE:
        print(f"{name}: {out_path} exists — skipping (set OVERWRITE = True to re-parse)")
        continue
    print(f"{name}: parsing ...", flush=True)
    raw = opener()
    try:
        st = pcap_to_flowtuple(raw, out_path, INTERVAL)
    finally:
        raw.close()
    run_stats.append({"pcap": name, **st})
    print(f"  link type {st['datalink']}, snaplen {st['snaplen']}")
    print(f"  packets read      : {st['packets']:>14,}   ({st['parse_seconds']} s, {st['packets_per_sec']:,} pkt/s)")
    print(f"    ipv4 aggregated : {st['ipv4']:>14,}   (of which non-first fragments: {st['fragments']:,})")
    print(f"    non-ipv4 skipped: {st['non_ipv4']:>14,}   {st['non_ipv4_ethertypes'] or ''}")
    print(f"    truncated       : {st['truncated']:>14,}")
    print(f"  flowtuple records : {st['flowtuple_records']:>14,}   "
          f"({st['ipv4'] / max(st['flowtuple_records'], 1):.2f} packets per record)")
    print(f"  capture span      : {_utc(st['first_ts'])} -> {_utc(st['last_ts'])} UTC")
    print(f"  wrote {out_path}  ({os.path.getsize(out_path) / 2**20:,.1f} MiB vs {size / 2**20:,.1f} MiB pcap)")

if run_stats:
    display(pd.DataFrame(run_stats).set_index("pcap").T)

---
# Empirical analysis

Everything below reads **only the Parquet files** — the pcaps are not touched again.
All "packets" figures are `packet_cnt`-weighted; "records" are FlowTuple rows.

Destination addresses here are the real greynet addresses, so tables below report
destinations as **counts and /24 aggregates**, not as individual addresses — keep it that
way if this notebook's output is going to be committed or shared.

## 6 · Load and overview

In [ ]:
import ipaddress

import matplotlib.pyplot as plt
import numpy as np

df = pd.concat((pd.read_parquet(p) for p in flowtuple_files), ignore_index=True)
df["bytes"] = df["ip_len"].astype("int64") * df["packet_cnt"]

PROTO_NAMES = {1: "ICMP", 6: "TCP", 17: "UDP", 47: "GRE", 50: "ESP", 41: "IPv6-in-IPv4", 132: "SCTP"}
df["proto"] = df["protocol"].map(PROTO_NAMES).fillna(df["protocol"].astype(str))


def ip_str(v):
    return str(ipaddress.ip_address(int(v)))


def net_str(v, bits):
    return f"{ip_str(int(v) << (32 - bits))}/{bits}"


total_packets = int(df["packet_cnt"].sum())
t_first, t_last = int(df["time"].min()), int(df["time"].max()) + INTERVAL
span = t_last - t_first

overview = pd.Series(
    {
        "flowtuple files": len(flowtuple_files),
        "time bins covered (UTC)": f"{_utc(t_first)} -> {_utc(t_last)}",
        "span (s)": span,
        "flowtuple records": f"{len(df):,}",
        "packets": f"{total_packets:,}",
        "bytes (IP total length)": f"{int(df['bytes'].sum()):,}",
        "mean packets / s": f"{total_packets / span:,.1f}",
        "mean packet size (bytes)": f"{df['bytes'].sum() / total_packets:,.1f}",
        "unique source IPs": f"{df['src_ip'].nunique():,}",
        "unique source /24s": f"{pd.unique(df['src_ip'].values >> 8).size:,}",
        "unique destination IPs": f"{df['dst_ip'].nunique():,}",
        "unique destination /24s": f"{pd.unique(df['dst_ip'].values >> 8).size:,}",
        "unique destination /16s": f"{pd.unique(df['dst_ip'].values >> 16).size:,}",
        "packets flagged masscan": f"{int(df.loc[df['is_masscan'], 'packet_cnt'].sum()):,}",
    },
    name="value",
)
display(overview.to_frame())

## 7 · Protocol mix and traffic over time

Per protocol: packets, bytes, records, and how many distinct sources and destinations
it involves. Packets and distinct sources often rank protocols differently — a handful
of heavy scanners can dominate packets while most *sources* arrive over another protocol.

In [ ]:
proto_tbl = df.groupby("proto").agg(
    packets=("packet_cnt", "sum"),
    bytes=("bytes", "sum"),
    records=("packet_cnt", "size"),
    uniq_src_ips=("src_ip", "nunique"),
    uniq_dst_ips=("dst_ip", "nunique"),
)
proto_tbl.insert(1, "pct_packets", (100 * proto_tbl["packets"] / total_packets).round(2))
proto_tbl = proto_tbl.sort_values("packets", ascending=False)
display(proto_tbl)

In [ ]:
# Chart style: one restrained look for every figure below.
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
PROTO_COLORS = {"TCP": BLUE, "UDP": ORANGE, "ICMP": AQUA}  # colour follows the protocol, everywhere
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": "#c9c8c2", "axes.labelcolor": "#52514e",
        "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": "#0b0b0b",
        "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "font.size": 9, "figure.dpi": 110,
    }
)

per_bin = df.pivot_table(index="time", columns="proto", values="packet_cnt", aggfunc="sum", fill_value=0)
main = [p for p in ("TCP", "UDP", "ICMP") if p in per_bin]
other = per_bin.drop(columns=main).sum(axis=1)
x = pd.to_datetime(per_bin.index, unit="s", utc=True)

fig, ax = plt.subplots(figsize=(9, 3.2))
for p in main:
    ax.plot(x, per_bin[p] / INTERVAL, color=PROTO_COLORS[p], lw=2, marker="o", ms=4, label=p)
if other.sum():
    ax.plot(x, other / INTERVAL, color=GRAY, lw=2, marker="o", ms=4, label="other")
ax.set_yscale("log")
ax.set_ylabel("packets / s (log)")
ax.set_xlabel(f"start of {INTERVAL} s bin (UTC)")
ax.set_title("Packet rate by protocol")
ax.legend(frameon=False, ncol=4, loc="upper left", bbox_to_anchor=(0, -0.22))
plt.show()

## 8 · Source and destination addresses

**Sources** — who is sending. The top-sources table shows, per source, how many distinct
greynet addresses and ports it touched: many destinations × one port is a horizontal
scan; few destinations × many ports is a vertical scan.

**Destinations** — how evenly the monitored addresses are hit. In a darknet, coverage is
close to uniform; in a greynet, unevenness across addresses is one of the things worth
comparing.

In [ ]:
src = df.groupby("src_ip").agg(
    packets=("packet_cnt", "sum"),
    uniq_dst_ips=("dst_ip", "nunique"),
    uniq_dst_ports=("dst_port", "nunique"),
    protocols=("proto", lambda s: ",".join(sorted(set(s)))),
)
dst_pkts = df.groupby("dst_ip")["packet_cnt"].sum()

qs = [0.5, 0.9, 0.99, 1.0]
dist = pd.DataFrame(
    {
        "packets per source IP": src["packets"].quantile(qs),
        "destination IPs per source IP": src["uniq_dst_ips"].quantile(qs),
        "packets per destination IP": dst_pkts.quantile(qs),
    }
)
dist.index = ["median", "p90", "p99", "max"]
display(dist.T)

n1 = int((src["packets"] == 1).sum())
top10_share = 100 * src["packets"].nlargest(10).sum() / total_packets
print(f"sources that sent exactly one packet : {n1:,} of {len(src):,} ({100 * n1 / len(src):.1f}%)")
print(f"share of packets from the top 10 sources: {top10_share:.1f}%")

top_src = src.sort_values("packets", ascending=False).head(TOP_N).copy()
top_src.insert(1, "pct_packets", (100 * top_src["packets"] / total_packets).round(2))
top_src.index = top_src.index.map(ip_str)
display(top_src)

In [ ]:
# Destinations: /24 aggregates only (see the note at the top of the analysis).
dst24 = df.assign(dst_net24=df["dst_ip"].values >> 8).groupby("dst_net24").agg(
    packets=("packet_cnt", "sum"),
    dst_ips_hit=("dst_ip", "nunique"),
    uniq_src_ips=("src_ip", "nunique"),
)
dst24.insert(1, "pct_packets", (100 * dst24["packets"] / total_packets).round(2))
top_dst24 = dst24.sort_values("packets", ascending=False).head(TOP_N)
top_dst24.index = top_dst24.index.map(lambda v: net_str(v, 24))
print(f"{len(dst24):,} destination /24s; top {len(top_dst24)} by packets:")
display(top_dst24)

# Fan-out: how many greynet addresses does each source touch?
fan = np.sort(src["uniq_dst_ips"].values)
ccdf = 1 - np.arange(len(fan)) / len(fan)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.step(fan, ccdf, where="post", color=BLUE, lw=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("distinct destination IPs contacted by a source (log)")
ax.set_ylabel("fraction of sources ≥ x (log)")
ax.set_title("Source fan-out (CCDF)")
plt.show()

## 9 · TCP — flags, destination ports, source ports

**Flags first**, because they decide how to read the ports:

- **SYN only** — connection attempts: scanning / probing. The **destination** port is the
  service being looked for; the source port is ephemeral and mostly noise.
- **SYN+ACK, RST, RST+ACK** — *backscatter*: replies from hosts that were sent packets
  spoofed with a greynet source address (DoS victims, typically). Here the **source**
  port is the attacked service, and the destination port is the noise.

So destination ports are tabulated over SYN-only traffic and all TCP, and source ports
are shown split by scan vs. backscatter.

In [ ]:
TCP_FLAG_BITS = ((0x02, "S"), (0x10, "A"), (0x01, "F"), (0x04, "R"), (0x08, "P"), (0x20, "U"), (0x40, "E"), (0x80, "C"))


def tcp_flags_str(flags):
    return "".join(letter for bit, letter in TCP_FLAG_BITS if flags & bit) or "."


tcp = df[(df["protocol"] == 6)].copy()
tcp["flags"] = tcp["tcp_flags"].map(tcp_flags_str)
tcp_packets = int(tcp["packet_cnt"].sum())

flag_tbl = tcp.groupby("flags").agg(packets=("packet_cnt", "sum"), uniq_src_ips=("src_ip", "nunique"))
flag_tbl.insert(1, "pct_tcp_packets", (100 * flag_tbl["packets"] / max(tcp_packets, 1)).round(2))
display(flag_tbl.sort_values("packets", ascending=False).head(12))

syn = tcp[tcp["tcp_flags"] == 0x02]
backscatter = tcp[tcp["flags"].isin(["SA", "R", "AR"])]
print(f"TCP packets            : {tcp_packets:,}")
print(f"  SYN-only (scanning)  : {int(syn['packet_cnt'].sum()):,}"
      f"   of which masscan-signature: {int(syn.loc[syn['is_masscan'], 'packet_cnt'].sum()):,}")
print(f"  SA / R / RA (backscatter-like): {int(backscatter['packet_cnt'].sum()):,}")
print(f"distinct TCP destination ports: {tcp['dst_port'].nunique():,}   distinct TCP source ports: {tcp['src_port'].nunique():,}")

In [ ]:
SERVICES = {
    21: "ftp", 22: "ssh", 23: "telnet", 25: "smtp", 53: "dns", 80: "http", 81: "http-alt", 110: "pop3",
    123: "ntp", 135: "msrpc", 137: "netbios-ns", 139: "netbios", 143: "imap", 161: "snmp", 389: "ldap",
    443: "https", 445: "smb", 465: "smtps", 500: "ike", 587: "submission", 993: "imaps", 995: "pop3s",
    1433: "mssql", 1521: "oracle", 1900: "ssdp", 2222: "ssh-alt", 2323: "telnet-alt", 3306: "mysql",
    3389: "rdp", 3478: "stun", 5060: "sip", 5353: "mdns", 5432: "postgres", 5555: "adb", 5900: "vnc",
    6379: "redis", 7547: "tr-069", 8080: "http-alt", 8443: "https-alt", 8728: "mikrotik", 9200: "elasticsearch",
    11211: "memcached", 27017: "mongodb", 37215: "huawei-hg532", 52869: "realtek-upnp",
}


def port_table(d, col, n=TOP_N):
    "Top-n values of a port column by packets, with distinct source / destination counts."
    g = d.groupby(col).agg(
        packets=("packet_cnt", "sum"),
        uniq_src_ips=("src_ip", "nunique"),
        uniq_dst_ips=("dst_ip", "nunique"),
    )
    g.insert(1, "pct_packets", (100 * g["packets"] / max(g["packets"].sum(), 1)).round(2))
    g = g.sort_values("packets", ascending=False).head(n)
    g.insert(0, "service", [SERVICES.get(int(p), "") for p in g.index])
    return g


print("TCP destination ports — SYN-only (what scanners are looking for):")
tcp_dport_syn = port_table(syn, "dst_port")
display(tcp_dport_syn)
print("TCP destination ports — all TCP:")
display(port_table(tcp, "dst_port"))

In [ ]:
if len(tcp_dport_syn):
    d = tcp_dport_syn.iloc[::-1]
    labels = [f"{p}  {s}".rstrip() for p, s in zip(d.index, d["service"])]
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.28 * len(d) + 1.2), sharey=True)
    for ax, col, title in (
        (axes[0], "packets", "SYN packets"),
        (axes[1], "uniq_src_ips", "distinct source IPs"),
    ):
        ax.barh(labels, d[col], color=BLUE, height=0.7)
        ax.set_title(title)
        ax.grid(axis="y", visible=False)
        ax.ticklabel_format(axis="x", style="plain")
    axes[0].set_ylabel("TCP destination port")
    fig.suptitle(f"Top {len(d)} scanned TCP ports — by volume, and by how many sources ask", x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    plt.show()
else:
    print("no SYN-only TCP traffic")

In [ ]:
print("TCP source ports — SYN-only (scanner side; fixed source ports are a tool fingerprint):")
display(port_table(syn, "src_port"))
print("TCP source ports — backscatter-like SA / R / RA (the service that was attacked):")
display(port_table(backscatter, "src_port"))

## 10 · UDP ports and ICMP types

For ICMP the FlowTuple `src_port` / `dst_port` fields carry **type** / **code**
(8/0 echo request = ping scanning; 0/0 echo reply, 3/x unreachable, 11/x time exceeded
= backscatter).

In [ ]:
udp = df[df["protocol"] == 17]
print(f"UDP packets: {int(udp['packet_cnt'].sum()):,}   distinct destination ports: {udp['dst_port'].nunique():,}"
      f"   distinct source ports: {udp['src_port'].nunique():,}")
print("UDP destination ports:")
display(port_table(udp, "dst_port"))
print("UDP source ports:")
display(port_table(udp, "src_port"))

ICMP_TYPES = {0: "echo reply", 3: "dest unreachable", 5: "redirect", 8: "echo request", 11: "time exceeded",
              13: "timestamp request", 14: "timestamp reply", 17: "address mask request"}
icmp = df[df["protocol"] == 1]
icmp_tbl = icmp.groupby(["src_port", "dst_port"]).agg(
    packets=("packet_cnt", "sum"), uniq_src_ips=("src_ip", "nunique"), uniq_dst_ips=("dst_ip", "nunique")
)
icmp_tbl.index.names = ["icmp_type", "icmp_code"]
icmp_tbl.insert(0, "meaning", [ICMP_TYPES.get(int(t), "") for t, _ in icmp_tbl.index])
print("ICMP type / code:")
display(icmp_tbl.sort_values("packets", ascending=False).head(TOP_N))

## 11 · TTL and packet sizes

Observed TTL = sender's initial TTL (almost always 64, 128, or 255) minus hop count, so
rounding up to the next of those gives a rough OS-family / tool split (64 Linux & most
IoT, 128 Windows, 255 network gear and some raw-socket scanners). Packet sizes are
dominated by a few values: 40 = bare TCP SYN with no options (stateless scanners such as
masscan / ZMap), 44–60 = SYN with options (real OS stacks).

In [ ]:
ttl = df.groupby("ttl")["packet_cnt"].sum().reindex(range(256), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(ttl.index, ttl.values, width=1.0, color=BLUE)
for t in (64, 128, 255):
    ax.axvline(t + 0.5, color=GRAY, lw=0.8, ls=":")
    ax.text(t, ax.get_ylim()[1], f" {t}", va="top", ha="right", color="#52514e")
ax.set_xlim(0, 256)
ax.set_xlabel("IP TTL as received")
ax.set_ylabel("packets")
ax.set_title("TTL distribution")
plt.show()

init_ttl = pd.cut(df["ttl"], [-1, 32, 64, 128, 255], labels=["≤32", "33–64 (init 64)", "65–128 (init 128)", "129–255 (init 255)"])
ttl_tbl = df.groupby(init_ttl, observed=False).agg(packets=("packet_cnt", "sum"), uniq_src_ips=("src_ip", "nunique"))
ttl_tbl.insert(1, "pct_packets", (100 * ttl_tbl["packets"] / total_packets).round(2))
display(ttl_tbl)

size_tbl = df.groupby(["proto", "ip_len"])["packet_cnt"].sum().rename("packets").reset_index()
size_tbl["pct_packets"] = (100 * size_tbl["packets"] / total_packets).round(2)
print("most common IP packet lengths:")
display(size_tbl.sort_values("packets", ascending=False).head(TOP_N).set_index(["proto", "ip_len"]))

syn_fp = syn.groupby(["tcp_synlen", "tcp_synwinlen"]).agg(packets=("packet_cnt", "sum"), uniq_src_ips=("src_ip", "nunique"))
print("most common SYN (TCP header length, window) fingerprints:")
display(syn_fp.sort_values("packets", ascending=False).head(10))

## 12 · Summary

In [ ]:
def _top(tbl, label):
    if not len(tbl):
        return f"{label}: none"
    items = [f"{idx}{'/' + r['service'] if r.get('service') else ''} ({r['pct_packets']:.1f}%)" for idx, r in tbl.head(5).iterrows()]
    return f"{label}: " + ", ".join(items)


print(f"{total_packets:,} IPv4 packets in {len(df):,} FlowTuple records over {span} s "
      f"({total_packets / span:,.0f} pkt/s)")
print(f"{df['src_ip'].nunique():,} source IPs -> {df['dst_ip'].nunique():,} destination IPs "
      f"in {len(dst24):,} /24s")
print("protocols: " + ", ".join(f"{p} {r['pct_packets']:.1f}%" for p, r in proto_tbl.head(4).iterrows()))
print(_top(tcp_dport_syn, "top scanned TCP ports (share of SYNs)"))
print(_top(port_table(backscatter, "src_port"), "top TCP backscatter source ports"))
print(_top(port_table(udp, "dst_port"), "top UDP destination ports (share of UDP)"))

## Next steps

- **Normalize by monitored addresses.** `sdscgreynetlist` gives the greynet address set;
  join it against `dst_ip` to get packets per monitored address (the number comparable to
  the darknet) and to see whether any destinations fall *outside* the list.
- **Enrich sources** with `prefix2asn` and country (RouteViews RIB + GeoLite2, as in
  `nids-telescope-traffic/flow_analysis.ipynb`) to complete the FlowTuple fields.
- **Scale out.** One pcap is single-threaded Python here. For days of data, run
  `pcap_to_flowtuple` once per pcap as a Slurm array (each file is independent), then
  read the Parquet directory with Spark or DuckDB.
- **Roll up to FlowTuple v4** if the darknet comparison needs identical records:
  `groupby(time, src_ip, dst_ip >> 8, dst_port, protocol)` with `nunique` counters.